In [224]:
#imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from ipywidgets import interact, FloatSlider, Output, VBox
import ipywidgets as widgets
from IPython.display import clear_output, display


In [225]:
#Gompertz model

#Model
def Gompertz(t, k1, k2, k3, k4, k5, k6):
    N_0 = N[0]
    return np.log(N_0) + k1 * np.exp(-np.exp(-k2 * (t - k3))) - k4 * np.exp(-np.exp(-k5 * (t - k6)))
    
# Estimate initial values for Gompertz
def estimate_initial_gompertz(t, N):
    N_0 = N[0]
    N_max = np.max(N)

    k1_init = np.log(N_max / N_0)
    k2_init = 1 / (t[np.argmax(np.diff(N))] - t[0])
    k3_init = t[np.argmax(np.diff(N))] / 2
    k4_init = k1_init / 2
    k5_init = k2_init
    k6_init = k3_init
    return [k1_init, k2_init, k3_init, k4_init, k5_init, k6_init]
    

In [226]:
#Churchill

#Model
def Churchill(t, k1, k2, k3, k4):
    return ((1 / k1) * np.exp(k3 * t) + (1 / k2) * np.exp(k4 * t)) ** (-1)

# Estimate initial values for Churchill
def estimate_initial_churchill(t, N):
    N_0 = N[0]
    N_max = np.max(N)
    if N_max == N_0:
        k1_init = 1.0
        k2_init = 1.0
    else:
        k1_init = 1 / (N_max - N_0)
        k2_init = 1 / (N_max - N_0)
    growth_phase = t[:len(t)//2]
    decay_phase = t[len(t)//2:]
    if np.ptp(growth_phase) == 0 or np.ptp(decay_phase) == 0:
        k3_init = 1.0
        k4_init = 1.0
    else:
        k3_init = np.log(N_max / N_0) / np.ptp(growth_phase)
        k4_init = np.log(N_max / N_0) / np.ptp(decay_phase)
    return [k1_init, k2_init, k3_init, k4_init]


In [227]:
# Weibull 

# Model
def Weibull(t, k1, k2, k3):
    return k1 * np.exp(-(t / k2) ** k3)

# Estimate initial values for Weibull
def estimate_initial_weibull(t, N):
    N_max = np.max(N)
    N_min = np.min(N)
    k1_init = N_max
    k2_init = np.mean(t)
    k3_init = 1.0
    return [k1_init, k2_init, k3_init]

In [228]:
#Logistic

#Model
def Logistic(t, k1, k2, k3):
    return k1 / (1 + np.exp(-k2 * (t - k3)))

# Estimate initial values for Logistic
def estimate_initial_logistic(t, N):
    N_max = np.max(N)
    N_min = np.min(N)
    k1_init = N_max - N_min
    k2_init = 1.0
    k3_init = np.median(t)
    return [k1_init, k2_init, k3_init]

In [229]:
#Logistic_decay

#Model
def Logistic_decay(t, k1, k2, k3):
    return k1 / (1 + np.exp(k2 * (t - k3)))

# Estimate initial values for Logistic_decay
def estimate_initial_logistic_decay(t, N):
    N_max = np.max(N)
    N_min = np.min(N)
    k1_init = N_max - N_min
    k2_init = 1.0
    k3_init = np.median(t)
    return [k1_init, k2_init, k3_init]

In [230]:
# Plotting function
def plot_fit(k1, k2, k3, k4, k5, k6, func, num_params):
    plt.scatter(t, N, label='Data')
    if func == Gompertz:
        plt.plot(t, np.exp(func(t, k1, k2, k3, k4, k5, k6)), 'r-', label='Fit')
    elif func == Churchill:
        plt.plot(t, func(t, k1, k2, k3, k4), 'r-', label='Fit')
    elif func in [Weibull, Logistic, Logistic_decay]:
        plt.plot(t, func(t, k1, k2, k3), 'r-', label='Fit')
    plt.xlim(0, 160)
    plt.ylim(0, 8)
    plt.xticks(np.arange(0, 180, step=20))
    plt.xlabel('t (days)')
    plt.ylabel('log(CFU/ml)')
    plt.legend()
    plt.show()

# Processes file and launches interactables
def process_file(file_name, func):
    clear_output(wait=True)
    global t, N
    data = pd.read_csv(file_name, header=None)
    t = data[0]
    N = data[1]

    #Initializing parameters
    if func == Gompertz:
        p0 = estimate_initial_gompertz(t, N)
        num_params = 6
    elif func == Churchill:
        p0 = estimate_initial_churchill(t, N)
        num_params = 4
    elif func == Logistic:
        p0 = estimate_initial_logistic(t, N)
        num_params = 3
    elif func == Logistic_decay:
        p0 = estimate_initial_logistic_decay(t, N)
        num_params = 3
    elif func == Weibull:
        p0 = estimate_initial_weibull(t,N)
        num_params=3
    else:
        p0 = [1, 1, 1, 1, 1, 1]
        num_params = 6

    print("Initial parameter estimates:", p0)

    try:
        if func in [Weibull, Logistic, Logistic_decay]:
            popt, pcov = curve_fit(func, t, N, p0=p0[:num_params], bounds=([0]*num_params, [np.inf]*num_params), maxfev=9999)
        else:
            popt, pcov = curve_fit(func, t, np.log(N), p0=p0[:num_params], bounds=(0, np.inf), maxfev=9999)
        print("Fitted parameters:", popt)
    except RuntimeError as e:
        print(f"An error occurred during fitting: {e}")
        return
    except ValueError as e:
        print(f"Value error during fitting: {e}")
        return

    sliders = [FloatSlider(min=0.01, max=100, step=0.001, value=param, description=f'k{i+1}', continuous_update=False) for i, param in enumerate(popt[:num_params])]

    out = Output()

    def update_plot(**kwargs):
        with out:
            clear_output(wait=True)
            plot_fit(kwargs.get('k1', 0), kwargs.get('k2', 0), kwargs.get('k3', 0), kwargs.get('k4', 0), kwargs.get('k5', 0), kwargs.get('k6', 0), func, num_params)

    interact(update_plot, **{f'k{i+1}': sliders[i] for i in range(num_params)})

    box = VBox([out], layout=widgets.Layout(height='500px'))
    display(box)

# Example usage
# process_file('your_data_file.csv', Gompertz)

In [231]:
process_file("control.csv", Gompertz)

Initial parameter estimates: [1.2568170281019202, 0.3333333333333333, 1.5, 0.6284085140509601, 0.3333333333333333, 1.5]
Fitted parameters: [ 1.17643862  2.24115774  6.25559615  0.2449459   0.07034968 68.26995583]


interactive(children=(FloatSlider(value=1.1764386217062868, continuous_update=False, description='k1', min=0.0…

In [234]:
process_file('B281.csv', Weiball)


Initial parameter estimates: [1.0, 1.0, 0.0, 0.0]
Fitted parameters: [1.96686819e+00 2.45624065e+05 9.94031417e-04 5.49814705e-02]


interactive(children=(FloatSlider(value=1.9668681890345567, continuous_update=False, description='k1', min=0.0…